# PART 03 · 1장 작업형 2 문제 구조

교재 194~210쪽 본문 소스 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

## 01 작업형 2 출제 방식

### 1 작업형 2가 만드는 결과와 풀이 흐름

#### 1. 작업형 2에서 만드는 결과

#### 2. 전체 풀이 흐름

### 2 기본 준비와 문제 파악

#### 1. 기본 파일 불러오기

이 장의 기본 예제로 온라인 학습 서비스의 다음 달 수강 연장 여부 데이터를 사용한다. 다음 코드는 이 장의 공통 예제로 사용하며, 이후 항목에서는 여기에서 불러온 train, test, submit을 활용한다.

In [1]:
import pandas as pd
base = 'data/part03_type2/binary_learning_renewal'
train = pd.read_csv(base+'/train.csv')
test = pd.read_csv(base+'/test.csv')
submit = pd.read_csv(base+'/sample_submission.csv')

#### 2. 문제에서 먼저 찾을 정보

## 02 train 데이터와 test 데이터

### 1 세 파일의 역할과 구조 확인

#### 1. 세 파일의 역할

#### 2. 파일 크기 비교

세 파일을 불러와 크기를 비교해본다.

In [2]:
print(train.shape)
print(test.shape)
print(submit.shape)

(720, 15)
(240, 14)
(240, 1)


#### 3. train 데이터 확인

train에는 모델이 참고할 입력 컬럼과 정답 컬럼이 함께 들어 있다. 이번 데이터에서 renew_next_month는 다음 달 수강 연장 여부를 나타내는 target이다.

In [3]:
cols = ['student_id', 'age_group', 'region', 'course_category',
        'quiz_avg_score', 'renew_next_month']
print(train[cols].head(3).to_string(index=False))

 student_id age_group region course_category  quiz_avg_score  renew_next_month
     100001       30대    강원권             SQL            69.5                 0
     100002       30대    수도권             SQL            72.8                 1
     100003       20대    수도권            머신러닝            65.0                 0


#### 4. test 데이터 확인

test는 최종 예측 대상 데이터이므로 renew_next_month가 없다. test에 target이 없는 상태가 정상이며, train의 정답을 test에 붙여서는 안 된다.

In [4]:
cols = ['student_id', 'age_group', 'region', 'course_category',
        'quiz_avg_score']
print(test[cols].head(3).to_string(index=False))

 student_id age_group region course_category  quiz_avg_score
     200001       20대    강원권            클라우드            95.0
     200002       30대    충청권          Python            64.2
     200003       40대    수도권          Python            72.0


### 2 train과 test의 컬럼 맞추기

#### 1. train과 test의 컬럼 차이

집합 연산을 사용하면 두 파일에서 서로 다른 컬럼을 빠르게 확인할 수 있다. train에만 있는 컬럼이 target 후보인지 문제 문장과 함께 확인한다.

In [5]:
print(set(train.columns) - set(test.columns))
print(set(test.columns) - set(train.columns))

{'renew_next_month'}
set()


#### 2. 입력 데이터의 구조 맞추기

target을 분리해 X, y, test의 구조를 나란히 확인해본다.

In [6]:
target = 'renew_next_month'
X = train.drop(columns=[target])
y = train[target]
print(X.shape)
print(y.shape)
print(test.shape)

(720, 14)
(720,)
(240, 14)


## 03 target 컬럼 확인

### 1 target 컬럼 찾기

#### 1. 입력 컬럼과 target 컬럼

#### 2. 컬럼 차이로 target 후보 찾기

#### 3. target 존재 여부 확인

target 컬럼이 실제로 train에만 있는지 코드로 확인해본다.

In [7]:
target = 'renew_next_month'
print(target in train.columns)
print(target in test.columns)

True
False


### 2 target 값 확인과 분리

#### 1. target 값과 고윳값 개수 확인

target의 값과 자료형, 고윳값 개수를 확인해본다.

In [8]:
print(train[[target]].head())
print(train[target].dtype)
print(train[target].nunique())

   renew_next_month
0                 0
1                 1
2                 0
3                 1
4                 0
int64
2


#### 2. target 분포 확인

분류 문제에서는 값의 종류뿐 아니라 각 클래스의 개수와 비율도 확인한다. 한 클래스가 지나치게 적으면 정확도만으로 모델을 평가하기 어려울 수 있다.

In [9]:
print(train[target].value_counts())
print(train[target].value_counts(normalize=True).round(3))

renew_next_month
1    388
0    332
Name: count, dtype: int64
renew_next_month
1    0.539
0    0.461
Name: proportion, dtype: float64


#### 3. target 분리와 구조 점검

target을 분리한 뒤 구조가 의도대로인지 점검해본다.

In [10]:
X = train.drop(columns=[target])
y = train[target]
print(X.shape)
print(y.shape)
print(target in X.columns)
print(set(X.columns) == set(test.columns))

(720, 14)
(720,)
False
True


## 04 분류와 회귀 구분

### 1 유형 판단 기준과 데이터 확인

#### 1. 문제 유형을 나누는 기준

#### 2. 회귀 데이터 불러오기

분류 데이터와 비교하기 위해 배달 소요 시간 데이터를 추가로 불러온다. pandas는 앞에서 이미 불러왔으므로 import 코드는 반복하지 않는다.

In [11]:
base_reg = 'data/part03_type2/regression_delivery_minutes'
reg_train = pd.read_csv(base_reg+'/train.csv')
reg_test = pd.read_csv(base_reg+'/test.csv')
reg_submit = pd.read_csv(base_reg+'/sample_submission.csv')
print(reg_train.shape)
print(reg_test.shape)
print(set(reg_train.columns)-set(reg_test.columns))

(840, 15)
(280, 14)
{'delivery_minutes'}


#### 3. target 값으로 유형 확인

두 데이터의 target 고윳값 개수와 분포를 코드로 확인해본다.

In [12]:
print('분류 target 고윳값: ', train['renew_next_month'].nunique())
print(train['renew_next_month'].value_counts().sort_index())
print('회귀 target 고윳값: ', reg_train['delivery_minutes'].nunique())
print(reg_train['delivery_minutes'].describe().round(1))

분류 target 고윳값:  2
renew_next_month
0    332
1    388
Name: count, dtype: int64
회귀 target 고윳값:  416
count    840.0
mean      53.3
std       13.6
min       21.6
25%       43.6
50%       52.9
75%       61.2
max      100.5
Name: delivery_minutes, dtype: float64


### 2 제출값 기준의 최종 판단

#### 1. 실젯값의 의미 비교

두 데이터의 앞부분을 나란히 출력해 target 값의 의미를 비교해본다.

In [13]:
print(train[['student_id', 'quiz_avg_score',
             'renew_next_month']].head(3).to_string(index=False))
print(reg_train[['order_id', 'distance_km', 'traffic_index',
                 'delivery_minutes']].head(3).to_string(index=False))

 student_id  quiz_avg_score  renew_next_month
     100001            69.5                 0
     100002            72.8                 1
     100003            65.0                 0
 order_id  distance_km  traffic_index  delivery_minutes
   500001         4.39           58.3              80.2
   500002         3.91           45.6              61.5
   500003         1.52           92.3              59.3


#### 2. 제출값과 모델 계열의 차이

#### 3. 분류와 회귀 판단 순서

## 05 이진분류와 다중분류 구분

### 1 클래스 개수로 구분하기

#### 1. target 고윳값 개수로 구분하기

#### 2. 다중분류 데이터 불러오기

이진분류의 train, test, submit은 앞에서 불러온 변수를 그대로 사용한다. 비교를 위해 고객 문의 티켓의 우선 처리 수준 데이터를 추가로 불러온다.

In [14]:
base_multi = 'data/part03_type2/multiclass_ticket_priority'
multi_train = pd.read_csv(base_multi+'/train.csv')
multi_test = pd.read_csv(base_multi+'/test.csv')
multi_submit = pd.read_csv(base_multi+'/sample_submission.csv')
print(multi_train.shape)
print(multi_test.shape)
print(set(multi_train.columns)-set(multi_test.columns))

(780, 15)
(260, 14)
{'priority_level'}


#### 3. 이진분류와 다중분류의 클래스 확인

두 데이터의 target 고윳값 개수와 클래스 분포를 코드로 확인해본다.

In [15]:
print('이진분류')
print(train['renew_next_month'].nunique())
print(train['renew_next_month'].value_counts().sort_index())
print('다중분류')
print(multi_train['priority_level'].nunique())
print(multi_train['priority_level'].value_counts().sort_index())

이진분류
2
renew_next_month
0    332
1    388
Name: count, dtype: int64
다중분류
3
priority_level
0    304
1    359
2    117
Name: count, dtype: int64


#### 4. 두 데이터의 샘플 비교

두 데이터의 앞부분을 출력해 target 값의 형태를 비교해본다.

In [16]:
print(train[['student_id', 'course_category',
             'renew_next_month']].head(3).to_string(index=False))
print(multi_train[['ticket_id', 'channel', 'product_area',
                   'priority_level']].head(3).to_string(index=False))

 student_id course_category  renew_next_month
     100001             SQL                 0
     100002             SQL                 1
     100003            머신러닝                 0
 ticket_id channel product_area  priority_level
    300001     app           계정               0
    300002   email          동영상               2
    300003     app          수료증               0


### 2 예측 방식과 코드 확인

#### 1. predict와 predict_proba의 역할

#### 2. 코드로 분류 유형 확인하기

고윳값 개수로 분류 유형을 판별하는 과정을 함수로 만들어 본다.

In [17]:
def get_class_type(data, target):
    class_count = data[target].nunique()
    if class_count == 2:
        return '이진분류'
    elif class_count >= 3:
        return '다중분류'
    else:
        return '확인 필요'
print(get_class_type(train, 'renew_next_month'))
print(get_class_type(multi_train, 'priority_level'))

이진분류
다중분류


## 06 제출 파일 구조

### 1 제출 파일 확인

#### 1. 제출 파일에서 확인할 내용

#### 2. sample_submission 확인

sample_submission의 크기와 컬럼, 값의 형태를 확인해본다.

In [18]:
print(test.shape)
print(submit.shape)
print(submit.columns.tolist())
print(submit.head())

(240, 14)
(240, 1)
['pred']
   pred
0   0.0
1   0.0
2   0.0
3   0.0
4   0.0


### 2 result.csv 만들기와 점검

#### 1. result.csv의 구조 먼저 만들기

아직 모델을 만들지 않았더라도 제출 파일의 구조를 먼저 점검할 수 있다. 다음 코드는 train target의 평균을 모든 test 행에 넣는다. 이는 파일 구조를 확인하기 위한 임시 예측값이며 모델 예측값은 아니다.

In [19]:
pred_value = train['renew_next_month'].mean()
result = pd.DataFrame({'pred': [pred_value]*len(test)})
print(pred_value)
print(result.shape)
print(result.head())

0.5388888888888889
(240, 1)
       pred
0  0.538889
1  0.538889
2  0.538889
3  0.538889
4  0.538889


#### 2. index=False로 저장하고 다시 읽기

판다스는 기본적으로 데이터프레임의 인덱스까지 CSV 파일에 저장한다. 제출 파일에는 예측 컬럼만 필요하므로 index=False를 지정한다. 저장이 끝나면 다시 읽어 실제 파일 구조를 확인한다.

In [20]:
result.to_csv('result.csv', index=False)
check = pd.read_csv('result.csv')
print(check.shape)
print(check.columns.tolist())
print(check.head())

(240, 1)
['pred']
       pred
0  0.538889
1  0.538889
2  0.538889
3  0.538889
4  0.538889


#### 3. 제출 전 결과 확인

저장한 파일을 다시 읽어 행 수, 컬럼, 결측치를 점검해본다.

In [21]:
print(len(check) == len(test))
print(list(check.columns) == list(submit.columns))
print(check.isnull().sum())

True
True
pred    0
dtype: int64
